In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx
import finufft

import cmasher as cmr
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

pa.settings.strict_units = False



In [2]:
# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/adiabatic-mhd/zoom4_ics_v1/output', 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/scratch/lperrone/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 
                   305, basename='snapshot')
center = snap.Cat.Group['GroupPos'][0]
widths = np.array([2e2, 2e2, 2e2], dtype=float)


Gauss = snap.uq('G')
arepo_length = snap.length
density_unit = snap['0_Density'].uq
L_box = (arepo_length*(widths[0]))

In [3]:

pe = tc.PotentialEnergy(snap, center, widths, 
                        snap['0_Coordinates'], snap['0_Masses'], npix=64)

In [4]:
np.sum(pe.tile.mass_per_tile)

array(467.34251913)

In [5]:
pe.tile.mass_per_tile.shape

(64, 64, 64)

In [6]:
np.sum(snap['0_Masses'][pe.index])

<PaicosQuantity 467.34251913 arepo_mass / small_h>

In [7]:
np.sum(pe.index)

541750

In [8]:
np.sum(pe.tile.particles_per_tile)

array(541750)

In [9]:
np.histogram(pe.tile.particles_per_tile.flatten())

(array([16474, 75632, 87223, 52010, 21477,  6909,  1864,   455,    82,
           18]),
 array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]))

In [10]:
# smoothing_length = 0.05*cp.asnumpy(pe.tile.tile_widths).min()*arepo_length
smoothing_length = np.cbrt((snap["0_Volume"]) / (4.0 * np.pi / 3.0))

smoothing_length_1 = snap['1_SubfindHsml']

smoothing_length_all = np.concatenate((smoothing_length, smoothing_length_1), axis=0)

Attempting to get derived variable: 0_Volume...	[DONE]



In [11]:
cp.asnumpy(pe.tile.tile_widths).min()*arepo_length

<PaicosQuantity 3.125 arepo_length small_a / small_h>

In [12]:
np.max(smoothing_length[pe.index])

<PaicosQuantity 2.35769007 small_a arepo_mass(1/3) / (arepo_density(1/3) small_h)>

In [13]:
angles = np.array([40, 25, 15, 10, 5, 0])

num_cells_away = 0.5*np.sqrt(3)/np.tan(0.5*angles*np.pi/180)

/tmp/ipykernel_711681/2012459274.py:3: RuntimeWarning: divide by zero encountered in divide
  num_cells_away = 0.5*np.sqrt(3)/np.tan(0.5*angles*np.pi/180)


In [14]:
num_cells_away

array([ 2.37938524,  3.90638815,  6.57811602,  9.89871566, 19.83524281,
               inf])

In [15]:
for angle in angles:
    U = pe.compute_potential(smoothing_length, angle=angle)
    print(f'theta: {angle:.1f} \t', 'U: ', U)

theta: 40.0 	 U:  -7.622727421541714e-05 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
theta: 25.0 	 U:  -7.622742179468868e-05 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
theta: 15.0 	 U:  -7.622770664374217e-05 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
theta: 10.0 	 U:  -7.622816394556577e-05 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
theta: 5.0 	 U:  -7.62302834580199e-05 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
theta: 0.0 	 U:  -7.623745571966832e-05 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)


In [16]:
snap_select = snap.select(pe.index, parttype=0)

In [17]:
def direct_calculation(snap, index, smoothing_length):
    snap_select = snap.select(index, parttype=0)
    Np = snap_select['0_Masses'].shape[0]

    if isinstance(smoothing_length.value, np.ndarray):
        smoothing_length = smoothing_length[index]
    else:
        smoothing_length = np.ones(Np) * smoothing_length

    potential = 0.0

    for i in range(Np):
        if (i%1000==0):
            print(f'particle {i}')
        h = smoothing_length[i].value
        for j in range(Np):
            m1m2 = (snap_select['0_Masses'][i]*snap_select['0_Masses'][j]).value
            if (i==j):
                # pass
                potential -= m1m2*(3./2.)/h
            else:
                r = np.sqrt(np.sum((snap_select['0_Coordinates'][i]-snap_select['0_Coordinates'][j])**2)).value
                potential_tmp = m1m2/r
                if (r <= h):
                    potential_tmp *= (0.5*r/h)*(3-(r/h)**2)
                potential -= potential_tmp
            

    G = 6.67430e-8 * snap_select.uq('cm^3 g^-1 s^-2')
    potential = 0.5 * G * potential * snap_select.mass**2 / snap_select.length
    return potential
    

In [18]:
from numba import njit, prange

@njit(parallel=True)
def parallel_potential(masses, pos, hsml):

    Np =  masses.shape[0]
    potential = np.zeros(Np, dtype=np.float64)
    for i in prange(Np):    
        h = hsml[i]
        for j in prange(Np):
            m1m2 = masses[i]*masses[j]
            if (i==j):
                potential[i] -= m1m2*(3./2.)/h
            else:
                r = np.sqrt(np.sum((pos[i]-pos[j])**2))
                potential_tmp = m1m2/r
                if (r <= h):
                    potential_tmp *= (0.5*r/h)*(3-(r/h)**2)
                potential[i] -= potential_tmp
                
    return potential


def direct_calculation_parallel(snap, positions, masses, smoothing_length):
    # snap_select = snap.select(index, parttype=0)

    Np = masses.shape[0]

    if isinstance(smoothing_length.value, np.ndarray):
        assert smoothing_length.shape == masses.shape
    else:
        smoothing_length = np.ones(Np) * smoothing_length

    mass = masses.value
    pos = positions.value
    hsml = smoothing_length.value

    potential = parallel_potential(mass, pos, hsml)

    potential = np.sum(potential)
    
    G = 6.67430e-8 * snap.uq('cm^3 g^-1 s^-2')
    potential = 0.5 * G * potential * snap.mass**2 / snap.length
    return potential
    

In [19]:
# U_direct = direct_calculation(snap, pe.index, smoothing_length)

In [20]:
# U_direct

In [21]:
# U_direct_parallel = direct_calculation_parallel(snap, snap['0_Coordinates'][pe.index], 
#                                                 snap['0_Masses'][pe.index], smoothing_length[pe.index])

In [22]:
# U_direct_parallel

In [23]:
# G = 6.67430e-8 * snap_select.uq('cm^3 g^-1 s^-2')
# print(0.5*G*np.sum(snap_select['0_Masses']**2*(3./2.)/smoothing_length))

In [24]:
# 0.5*G*cp.asnumpy(np.sum(pe.tile.mass_per_tile**2))*(3./2.)/smoothing_length

In [25]:
# direct_calculation_parallel(snap, all_pos[pe_all.index], 
#                             all_mass[pe_all.index], smoothing_length_all[pe_all.index])

In [26]:
from numba import njit, prange
import cupy as cp
from numba import cuda
import math


@cuda.jit()
def cuda_potential(masses, pos, hsml, potential):

    Np =  masses.shape[0]
    i = cuda.grid(1)

    if (i < Np):
        h = hsml[i]
        for j in range(Np):
            m1m2 = masses[i]*masses[j]
            if (i==j):
                potential[i] -= m1m2*(3./2.)/h
            else:
                r = math.sqrt((pos[i][0] - pos[j][0])**2 +
                             (pos[i][1] - pos[j][1])**2 +
                             (pos[i][2] - pos[j][2])**2)
                potential_tmp = m1m2/r
                if (r <= h):
                    potential_tmp *= (0.5*r/h)*(3-(r/h)**2)
                potential[i] -= potential_tmp
                


def direct_calculation_cuda(snap, positions, masses, smoothing_length):
    # snap_select = snap.select(index, parttype=0)

    Np = masses.shape[0]

    if isinstance(smoothing_length.value, np.ndarray):
        assert smoothing_length.shape == masses.shape
    else:
        smoothing_length = np.ones(Np) * smoothing_length

    mass = cp.array(masses.value)
    pos = cp.array(positions.value)
    hsml = cp.array(smoothing_length.value)
    potential = cp.zeros_like(mass)

    threadsperblock = 256
    blocks_1d = (Np + (threadsperblock - 1)) // threadsperblock
    

    cuda_potential[blocks_1d, threadsperblock](mass, pos, hsml, potential)

    potential = cp.asnumpy(cp.sum(potential))
    
    G = 6.67430e-8 * snap.uq('cm^3 g^-1 s^-2')
    potential = 0.5 * G * potential * snap.mass**2 / snap.length
    return potential
    

In [27]:
direct_calculation_cuda(snap, snap['0_Coordinates'][pe.index], 
                            snap['0_Masses'][pe.index], smoothing_length[pe.index])

<PaicosQuantity -7.62374557e-05 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)>

In [28]:
all_pos = np.concatenate((snap['0_Coordinates'], snap['1_Coordinates']), axis=0)
all_mass = np.concatenate((snap['0_Masses'], snap['1_Masses']), axis=0)



Attempting to get derived variable: 1_Masses...	[DONE]



In [29]:
pe_all = tc.PotentialEnergy(snap, center, widths, 
                        all_pos, all_mass, npix=64)

In [30]:
np.sum(pe_all.index)

2063896

In [31]:
np.histogram(pe_all.tile.particles_per_tile.flatten())

(array([260781,   1145,    127,     47,     17,     14,      4,      1,
             1,      7]),
 array([  0. ,  41.8,  83.6, 125.4, 167.2, 209. , 250.8, 292.6, 334.4,
        376.2, 418. ]))

In [32]:
np.max(smoothing_length_all[pe_all.index])

<PaicosQuantity 7.90964746 small_a arepo_mass(1/3) / (arepo_density(1/3) small_h)>

In [33]:
# %%timeit
# U_cuda_all = direct_calculation_cuda(snap, all_pos[pe_all.index], 
#                             all_mass[pe_all.index], smoothing_length_all[pe_all.index])


In [40]:
direct_calculation_cuda(snap, all_pos[pe_all.index], 
                            all_mass[pe_all.index], smoothing_length_all[pe_all.index])

<PaicosQuantity -0.01657191 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)>

In [37]:
%%timeit
print(pe_all.compute_potential(smoothing_length_all, angle=40))

-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
-0.016566980987276048 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)
3.52 s ± 12.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
print(U_all)

NameError: name 'U_all' is not defined

In [ ]:
%%timeit
U_all = pe_all.compute_potential(smoothing_length_all, angle=20)

In [ ]:
print(U_all)

In [ ]:
U_all = pe_all.compute_potential(smoothing_length_all, angle=10)
print(U_all)

In [38]:
pe_DM = tc.PotentialEnergy(snap, center, widths, 
                        snap['1_Coordinates'], snap['1_Masses'], npix=64)

In [39]:

direct_calculation_cuda(snap, snap['1_Coordinates'][pe_DM.index], 
                            snap['1_Masses'][pe_DM.index], snap['1_SubfindHsml'][pe_DM.index])

<PaicosQuantity -0.01442603 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)>